# VAE Hard Task — Advanced Multi-Modal Clustering

Core pipeline:
- **6 Models**: MLP-VAE · Beta-VAE · CVAE · Conv1D-VAE · Autoencoder · Multi-Modal VAE
- **2 Baselines**: PCA + KMeans · Spectral + KMeans
- **3 Datasets**: FMA · LMD · GTZAN — each augmented with real Bangla audio
- **6 Metrics**: Silhouette · Davies-Bouldin · CH · NMI · ARI · Cluster Purity

Advanced Extensions:
- **Ext-1** GMVAE — Gaussian Mixture VAE (K-component learned prior)
- **Ext-2** β-Sensitivity — Beta-VAE sweep over β ∈ {0.5, 1, 2, 4, 8, 16}
- **Ext-3** MIG — Mutual Information Gap disentanglement metric
- **Ext-4** SLERP — Spherical interpolation between genre centroids
- **Ext-5** Zero-Shot Transfer — Cross-dataset transfer with PCA alignment
- **Ext-A** Contrastive VAE — InfoNCE (SimCLR) + β-VAE with projection head
- **Ext-B** DANN-VAE — Domain Adversarial VAE with gradient reversal (Ganin 2016)

## 0. Setup

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

In [ ]:
import warnings
import numpy as np
import torch
from config.config import (
    BANGLA_DIR, BANGLA_QUERIES, BATCH_SIZE, BETA_VAE_B,
    EPOCHS, FMA_METADATA_URL, GENRE_VOCAB, GTZAN_URLS,
    HIDDEN_DIMS, KMEANS_NINIT, LANG_COLORS, LANG_MARKERS,
    LATENT_DIM, LMD_URL, LR, LYRIC_DIM, MODEL_COLORS, N_BANGLA_PER_GENRE,
)
from src.data.bangla import get_bangla
from src.data.fma import download_fma_metadata, load_fma
from src.data.gtzan import download_gtzan_csv, load_gtzan
from src.data.lmd import download_lmd, load_lmd

warnings.filterwarnings("ignore")
np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = PROJECT_ROOT / "results" / "hard"
RESULTS.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 1. Download + Load Datasets

In [ ]:
FMA_DIR   = PROJECT_ROOT / "data" / "fma" / "fma_metadata"
LMD_DIR   = PROJECT_ROOT / "data" / "lmd"
GTZAN_CSV = PROJECT_ROOT / "data" / "gtzan" / "features_30_sec.csv"

download_fma_metadata(FMA_DIR, FMA_METADATA_URL)
download_lmd(LMD_DIR, LMD_URL)
download_gtzan_csv(GTZAN_CSV, GTZAN_URLS)

X_fma, y_fma = load_fma(FMA_DIR)
X_lmd, y_lmd = load_lmd(LMD_DIR)
X_gh,  y_gh  = load_gtzan(GTZAN_CSV)

datasets_raw = []
for X_en, y_en, ds_name in [
    (X_fma, y_fma, "FMA (Free Music Archive)"),
    (X_lmd, y_lmd, "LMD (Lakh MIDI Dataset)"),
    (X_gh,  y_gh,  "GTZAN via GitHub"),
]:
    lang_en = np.array(["English"] * len(X_en))
    bX, by, bl = get_bangla(X_en.shape[1], BANGLA_QUERIES, BANGLA_DIR, N_BANGLA_PER_GENRE)
    if len(bX) > 0:
        X_m = np.vstack([X_en, bX]); y_m = np.concatenate([y_en, by]); l_m = np.concatenate([lang_en, bl])
    else:
        X_m, y_m, l_m = X_en, y_en, lang_en
    datasets_raw.append((X_m, y_m, l_m, ds_name))
    print(f"  {ds_name}: {X_m.shape}")

## 2. Run Full Hard Pipeline

In [ ]:
from scripts.run_hard import full_pipeline

ALL = {}
for X_raw, y_labels, lang_labels, ds_name in datasets_raw:
    ds_key = ds_name.split()[0]
    ALL[ds_key] = full_pipeline(
        X_raw, y_labels, lang_labels, ds_name,
        latent_dim=LATENT_DIM, epochs=EPOCHS,
        beta_vae_b=BETA_VAE_B, device=DEVICE, results_dir=RESULTS,
    )
print("All experiments complete!")

## 3. Metrics Table + Heatmap

In [ ]:
import pandas as pd
from src.visualization.plots import plot_metrics_heatmap

MODEL_KEYS = ["MLP-VAE","Beta-VAE","CVAE","Conv-VAE","AE","Multimodal","PCA","Spectral"]
rows = []
for ds_key, res in ALL.items():
    for mkey in MODEL_KEYS:
        for algo in ["KMeans","Agglomerative","DBSCAN"]:
            r = res["cl"][mkey][algo]
            rows.append({"Dataset": ds_key, "Model": mkey, "Algorithm": algo,
                "Silhouette": r["sil"], "Davies-Bouldin": r["db"],
                "Calinski-H": r["ch"], "NMI": r["nmi"], "ARI": r["ari"], "Purity": r["purity"]})
df_all = pd.DataFrame(rows)
df_all.to_csv(RESULTS / "full_metrics.csv", index=False)
plot_metrics_heatmap(df_all, save_path=RESULTS / "metrics_heatmap.png")
print(df_all[df_all.Algorithm=="KMeans"].to_string(index=False))

## 4. Analysis — Why VAE Outperforms Baselines

In [ ]:
sep = "=" * 70
print(sep)
print("  VAE vs Baselines — KMeans Silhouette")
print(sep)
for ds_key, res in ALL.items():
    pca_sil = res["cl"]["PCA"]["KMeans"]["sil"]
    for mkey in ["MLP-VAE","Beta-VAE","CVAE","Multimodal"]:
        v = res["cl"][mkey]["KMeans"]["sil"]
        delta = v - pca_sil
        verdict = "BETTER" if delta > 0.01 else ("WORSE" if delta < -0.01 else "SIMILAR")
        print(f"  {ds_key:<6} | {mkey:<12} vs PCA: {v:.4f} vs {pca_sil:.4f}  Δ={delta:+.4f}  [{verdict}]")
print()
print("Saved results:", RESULTS)

## 5. Advanced Extensions — Setup

Import all extension helpers and config.

In [ ]:
import pandas as pd
import torch
import numpy as np
from sklearn.manifold import TSNE
import umap as umap_lib
from sklearn.cluster import KMeans
from config.config import (
    BATCH_SIZE, BETA_VALUES, CONTRASTIVE_TEMPERATURE, DANN_COMMON_DIM,
    DANN_DOMAIN_WEIGHT, HIDDEN_DIMS, KMEANS_NINIT, LAMBDA_VALUES,
    LATENT_DIM, LR, N_INTERP, SWEEP_EPOCHS,
)
from src.models import (
    BetaVAE, ContrastiveVAE, GMVAE,
    build_dann_dataset, train_contrastive_vae, train_dann_vae, train_gmvae,
)
from src.clustering.engine import compute_metrics, compute_mig
from src.training.trainer import train_model
from src.visualization.plots import (
    plot_beta_sensitivity, plot_contrastive_results, plot_dann_results,
    plot_gmvae_results, plot_interpolation, plot_mega_heatmap,
    plot_mig_scores, plot_transfer_results,
)
print('Extension imports OK')

## Ext-1: GMVAE — Gaussian Mixture VAE

Replace the standard N(0,I) prior with a **K-component Gaussian mixture**.
Each component corresponds to one genre cluster. Jointly learns:
- Which component each sample belongs to (soft assignment)
- VAE reconstruction with genre-disentangled latent structure

In [ ]:
GMVAE_RESULTS = {}

for ds_key, res in ALL.items():
    print(f'\n  GMVAE — {res["name"]}')
    X_sc_d   = res['X_sc']
    y_true_d = res['y_true']
    K_d      = res['n_class']

    gm, gm_hist, _ = train_gmvae(
        X_sc_d, n_components=K_d, latent_dim=LATENT_DIM,
        h=HIDDEN_DIMS, epochs=EPOCHS, lr=LR,
        batch_size=BATCH_SIZE, device=DEVICE,
    )

    gm.eval()
    X_t = torch.FloatTensor(X_sc_d)
    Z_parts, QY_parts = [], []
    with torch.no_grad():
        for i in range(0, len(X_t), BATCH_SIZE):
            mu, lv, qy = gm.encode(X_t[i:i+BATCH_SIZE].to(DEVICE))
            Z_parts.append(mu.cpu().numpy())
            QY_parts.append(qy.cpu().numpy())
    Z_gm    = np.vstack(Z_parts)
    QY      = np.vstack(QY_parts)
    gm_lbls = QY.argmax(axis=1)

    m_gm = compute_metrics(Z_gm, y_true_d, gm_lbls)
    print(f'  Sil={m_gm["sil"]:.4f}  DB={m_gm["db"]:.4f}  '
          f'NMI={m_gm["nmi"]:.4f}  ARI={m_gm["ari"]:.4f}  Purity={m_gm["purity"]:.4f}')

    tsne_gm = TSNE(n_components=2, perplexity=40, max_iter=500, random_state=42).fit_transform(Z_gm)
    umap_gm = umap_lib.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(Z_gm)

    GMVAE_RESULTS[ds_key] = dict(
        Z=Z_gm, QY=QY, labels=gm_lbls, metrics=m_gm,
        hist=gm_hist, model=gm,
        tsne=tsne_gm, umap=umap_gm,
        y_true=y_true_d, y_labels=res['y_labels'],
        lang_labels=res['lang_labels'], le=res['le'], K=K_d,
    )

plot_gmvae_results(GMVAE_RESULTS, ALL, save_path=RESULTS / 'ext1_gmvae_results.png')
print('Saved: ext1_gmvae_results.png')

## Ext-2: β-Sensitivity Analysis

Train Beta-VAE with **β ∈ {0.5, 1, 2, 4, 8, 16}** on each dataset.
Plot the tradeoff between reconstruction quality, cluster separation, and disentanglement.

In [ ]:
BETA_RESULTS = {}

for ds_key, res in ALL.items():
    print(f'\n  β-sweep — {res["name"]}')
    X_sc_d   = res['X_sc']
    y_true_d = res['y_true']
    K_d      = res['n_class']
    X_t      = torch.FloatTensor(X_sc_d)
    BETA_RESULTS[ds_key] = {}

    for beta_v in BETA_VALUES:
        print(f'  beta={beta_v:.1f} ...', end=' ', flush=True)
        m_b = BetaVAE(X_sc_d.shape[1], LATENT_DIM, h=HIDDEN_DIMS).to(DEVICE)
        m_b, _, _ = train_model(
            X_sc_d, m_b, model_type='beta_vae', beta=beta_v,
            epochs=SWEEP_EPOCHS, lr=LR, batch_size=BATCH_SIZE,
            device=DEVICE, verbose=False,
        )
        m_b.eval()
        Z_list, recon_err, n_b = [], 0.0, 0
        with torch.no_grad():
            for i in range(0, len(X_t), BATCH_SIZE):
                bx = X_t[i:i+BATCH_SIZE].to(DEVICE)
                mu_b, _ = m_b.enc(bx)
                Z_list.append(mu_b.cpu().numpy())
                recon_b, _, _, _ = m_b(bx)
                recon_err += float(torch.nn.functional.mse_loss(recon_b, bx).item())
                n_b += 1
        Z_b   = np.vstack(Z_list)
        km_b  = KMeans(n_clusters=K_d, n_init=KMEANS_NINIT, random_state=42).fit(Z_b)
        met_b = compute_metrics(Z_b, y_true_d, km_b.labels_)
        met_b['recon_loss'] = recon_err / max(n_b, 1)
        print(f'Sil={met_b["sil"]:.3f}  NMI={met_b["nmi"]:.3f}  Recon={met_b["recon_loss"]:.3f}')
        BETA_RESULTS[ds_key][beta_v] = dict(metrics=met_b, Z=Z_b)

plot_beta_sensitivity(BETA_RESULTS, save_path=RESULTS / 'ext2_beta_sensitivity.png')
print('Saved: ext2_beta_sensitivity.png')

## Ext-3: MIG — Mutual Information Gap

The **Mutual Information Gap** (Chen et al., 2018) is the *de facto* standard
quantitative metric for disentanglement.

**Formula:** MIG = (top_MI - second_MI) / H(v)

MIG -> 1.0 means perfectly disentangled (one latent dim = one generative factor).

In [ ]:
MIG_RESULTS = {}
model_z_map = [
    ('MLP-VAE',  'mlp'),
    ('Beta-VAE', 'beta'),
    ('CVAE',     'cvae'),
    ('Conv-VAE', 'conv'),
    ('AE',       'ae'),
    ('PCA',      'pca'),
]

print('Computing MIG scores (higher = more disentangled)\n')
for ds_key, res in ALL.items():
    MIG_RESULTS[ds_key] = {}
    print(f'  Dataset: {ds_key}')
    for mname, zkey in model_z_map:
        mig_score, mi_dims = compute_mig(res['Z'][zkey], res['y_true'])
        MIG_RESULTS[ds_key][mname] = dict(mig=mig_score, mi_per_dim=mi_dims)
        print(f'    {mname:<12} MIG = {mig_score:.4f}')
    if ds_key in GMVAE_RESULTS:
        mig_gm, mi_gm = compute_mig(GMVAE_RESULTS[ds_key]['Z'], res['y_true'])
        MIG_RESULTS[ds_key]['GMVAE'] = dict(mig=mig_gm, mi_per_dim=mi_gm)
        print(f'    {"GMVAE":<12} MIG = {mig_gm:.4f}')

plot_mig_scores(MIG_RESULTS, save_path=RESULTS / 'ext3_mig_scores.png')
print('Saved: ext3_mig_scores.png')

## Ext-4: Latent Space Interpolation (SLERP)

**Spherical linear interpolation** between two genre centroids:
1. Compute mean latent code per genre: z_A, z_B
2. Interpolate: z(t) = SLERP(z_A, z_B, t) for t in [0, 1]
3. Decode z(t) to reconstructed feature vector

Smooth transitions prove the latent space is **semantically structured**.

In [ ]:
# plot_interpolation runs SLERP internally using the MLP-VAE model per dataset
plot_interpolation(ALL, n_interp=N_INTERP, save_dir=RESULTS)
print('Saved: interpolation_*.png')

## Ext-5: Cross-Dataset Transfer (Zero-Shot)

Train VAE on Dataset A. Encode Dataset B **without retraining**.

Tests if the latent space captures *universal* audio structure.

Pairs tested: FMA->LMD, FMA->GTZAN, LMD->GTZAN, GTZAN->FMA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def _align_features(X_src, X_tgt):
    src_dim, tgt_dim = X_src.shape[1], X_tgt.shape[1]
    if src_dim == tgt_dim:
        return X_tgt
    if tgt_dim > src_dim:
        return PCA(n_components=src_dim, random_state=42).fit_transform(X_tgt).astype('float32')
    pad = np.zeros((X_tgt.shape[0], src_dim - tgt_dim), dtype='float32')
    return np.hstack([X_tgt, pad])

transfer_pairs = [('FMA', 'LMD'), ('FMA', 'GTZAN'), ('LMD', 'GTZAN'), ('GTZAN', 'FMA')]
TRANSFER_RESULTS = {}

for src_key, tgt_key in transfer_pairs:
    if src_key not in ALL or tgt_key not in ALL:
        continue
    print(f'\n  {src_key} -> {tgt_key}')
    src_model = ALL[src_key]['models']['mlp']
    src_X     = ALL[src_key]['X_sc']
    tgt_X     = ALL[tgt_key]['X_sc']
    tgt_y     = ALL[tgt_key]['y_true']
    tgt_K     = ALL[tgt_key]['n_class']

    tgt_X_al = _align_features(src_X, tgt_X)
    tgt_X_sc = StandardScaler().fit_transform(tgt_X_al).astype('float32')

    src_model.eval()
    Z_tr_parts = []
    X_t_tr = torch.FloatTensor(tgt_X_sc)
    with torch.no_grad():
        for i in range(0, len(X_t_tr), BATCH_SIZE):
            mu_t, _ = src_model.enc(X_t_tr[i:i+BATCH_SIZE].to(DEVICE))
            Z_tr_parts.append(mu_t.cpu().numpy())
    Z_tr = np.vstack(Z_tr_parts)

    K_t  = min(tgt_K, len(Z_tr) - 1)
    km_t = KMeans(n_clusters=K_t, n_init=KMEANS_NINIT, random_state=42).fit(Z_tr)
    m_tr = compute_metrics(Z_tr, tgt_y, km_t.labels_)
    m_nat = ALL[tgt_key]['cl']['MLP-VAE']['KMeans']
    retain = m_tr['sil'] / (m_nat['sil'] + 1e-8)

    print(f'  Transfer  Sil={m_tr["sil"]:.4f}  NMI={m_tr["nmi"]:.4f}  Retention={retain*100:.1f}%')
    print(f'  Native    Sil={m_nat["sil"]:.4f}  NMI={m_nat["nmi"]:.4f}')
    key = f'{src_key}->{tgt_key}'
    TRANSFER_RESULTS[key] = dict(
        Z=Z_tr, labels=km_t.labels_, metrics=m_tr,
        native=m_nat, retention=retain,
        tgt_y=tgt_y, tgt_le=ALL[tgt_key]['le'],
    )

plot_transfer_results(TRANSFER_RESULTS, save_path=RESULTS / 'ext5_transfer_results.png')
print('Saved: ext5_transfer_results.png')

## Ext-A: Contrastive VAE (InfoNCE + β-VAE)

Combines β-VAE ELBO with **InfoNCE (NT-Xent / SimCLR)** loss.

- A projection head maps z to R^64 (L2-normalised) for contrastive loss
- Same-genre pairs treated as positives; cross-genre pairs in batch as negatives
- λ-sweep over {0.1, 0.5, 1.0} — best λ selected by Silhouette Score

In [ ]:
CVAE_CON_RESULTS = {}

for ds_key, res in ALL.items():
    print(f'\n  ContrastiveVAE — {res["name"]}')
    X_sc_d   = res['X_sc']
    y_lbl_d  = res['y_labels']
    y_true_d = res['y_true']
    K_d      = res['n_class']
    X_t      = torch.FloatTensor(X_sc_d)

    best_sil_c, best_lam_c = -1.0, LAMBDA_VALUES[0]
    best_model_c = best_Z_c = best_met_c = best_hist_c = None
    lambda_sweep = {}

    for lam_v in LAMBDA_VALUES:
        print(f'  lam={lam_v:.1f} ...', end=' ', flush=True)
        m_con, hist_con, _ = train_contrastive_vae(
            X_sc_d, y_lbl_d, latent_dim=LATENT_DIM, h=HIDDEN_DIMS,
            epochs=EPOCHS, lr=LR, beta=1.0,
            lam=lam_v, temperature=CONTRASTIVE_TEMPERATURE,
            batch_size=BATCH_SIZE, device=DEVICE,
        )
        m_con.eval()
        Z_list_c = []
        with torch.no_grad():
            for i in range(0, len(X_t), BATCH_SIZE):
                mu_c, _ = m_con.enc(X_t[i:i+BATCH_SIZE].to(DEVICE))
                Z_list_c.append(mu_c.cpu().numpy())
        Z_c   = np.vstack(Z_list_c)
        km_c  = KMeans(n_clusters=K_d, n_init=KMEANS_NINIT, random_state=42).fit(Z_c)
        met_c = compute_metrics(Z_c, y_true_d, km_c.labels_)
        print(f'Sil={met_c["sil"]:.4f}  NMI={met_c["nmi"]:.4f}')
        lambda_sweep[lam_v] = dict(metrics=met_c, Z=Z_c, model=m_con)
        if met_c['sil'] > best_sil_c:
            best_sil_c  = met_c['sil']; best_lam_c = lam_v
            best_model_c = m_con; best_Z_c = Z_c
            best_met_c = met_c; best_hist_c = hist_con

    print(f'  Best lam={best_lam_c:.1f}  Sil={best_sil_c:.4f}')
    tsne_c = TSNE(n_components=2, perplexity=40, max_iter=500, random_state=42).fit_transform(best_Z_c)
    umap_c = umap_lib.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(best_Z_c)

    CVAE_CON_RESULTS[ds_key] = dict(
        Z=best_Z_c, metrics=best_met_c, hist=best_hist_c,
        model=best_model_c, best_lam=best_lam_c,
        lambda_sweep=lambda_sweep,
        tsne=tsne_c, umap=umap_c,
        y_true=y_true_d, y_labels=y_lbl_d,
        lang_labels=res['lang_labels'], le=res['le'], K=K_d,
    )

plot_contrastive_results(CVAE_CON_RESULTS, ALL, save_path=RESULTS / 'extA_contrastive_vae.png')
print('Saved: extA_contrastive_vae.png')

## Ext-B: DANN-VAE — Domain Adversarial VAE

Shared encoder across FMA / LMD / GTZAN with a **domain classifier head**.

- Gradient reversal layer (Ganin et al. 2016) flips domain gradient during backprop
- Forces encoder to learn **domain-invariant** representations
- lambda progressively ramped from 0 to 1 over training epochs

In [ ]:
print('Building combined 3-domain dataset...')
X_dann, d_dann, y_dann, lang_dann, DS_INFO = build_dann_dataset(
    ALL, datasets_raw, common_dim=DANN_COMMON_DIM,
)
print(f'Domain distribution: {np.bincount(d_dann).tolist()}')

dann_model, dann_hist, dann_loss = train_dann_vae(
    X_dann, d_dann, latent_dim=LATENT_DIM, h=HIDDEN_DIMS,
    epochs=EPOCHS, lr=LR, beta=1.0,
    lam_domain=DANN_DOMAIN_WEIGHT, batch_size=BATCH_SIZE, device=DEVICE,
)
print(f'DANN-VAE best loss: {dann_loss:.4f}')

dann_model.eval()
X_t_dann = torch.FloatTensor(X_dann)
Z_dann_parts = []
with torch.no_grad():
    for i in range(0, len(X_t_dann), BATCH_SIZE):
        mu_d, _ = dann_model.enc(X_t_dann[i:i+BATCH_SIZE].to(DEVICE))
        Z_dann_parts.append(mu_d.cpu().numpy())
Z_dann_all = np.vstack(Z_dann_parts)

DANN_RESULTS = {}
offset = 0
for ds_key, d_id, le_d, y_true_d, n_samples in DS_INFO:
    Z_d = Z_dann_all[offset:offset+n_samples]
    K_d = ALL[ds_key]['n_class']
    offset += n_samples
    km_d  = KMeans(n_clusters=K_d, n_init=KMEANS_NINIT, random_state=42).fit(Z_d)
    met_d = compute_metrics(Z_d, y_true_d, km_d.labels_)
    nat_d = ALL[ds_key]['cl']['MLP-VAE']['KMeans']
    delta_s = met_d['sil'] - nat_d['sil']
    msg = 'domain-inv helps' if delta_s > 0 else 'domain-spec better'
    print(f'  {ds_key}  DANN Sil={met_d["sil"]:.4f}  '
          f'Native Sil={nat_d["sil"]:.4f}  Delta={delta_s:+.4f}  ({msg})')
    tsne_d = TSNE(n_components=2, perplexity=40, max_iter=500, random_state=42).fit_transform(Z_d)
    DANN_RESULTS[ds_key] = dict(
        Z=Z_d, metrics=met_d, labels=km_d.labels_,
        tsne=tsne_d, y_true=y_true_d,
        lang_labels=ALL[ds_key]['lang_labels'],
        le=le_d, K=K_d,
    )

tsne_full = TSNE(n_components=2, perplexity=40, max_iter=500, random_state=42).fit_transform(Z_dann_all)
DANN_RESULTS['_full'] = dict(Z=Z_dann_all, tsne=tsne_full, d_labels=d_dann, y_dann=y_dann)

d_acc = plot_dann_results(
    DANN_RESULTS, dann_hist, X_dann, d_dann, dann_model,
    ALL, DEVICE, batch_size=BATCH_SIZE,
    save_path=RESULTS / 'extB_dann_vae_results.png',
)
print(f'Saved: extB_dann_vae_results.png  (domain classifier acc={d_acc:.1f}%)')

## 6. Mega Comparison Heatmap

All 11 models x 4 metrics x 3 datasets — unified comparison table and heatmap.

In [ ]:
MODEL_KEYS_ALL = ['MLP-VAE', 'Beta-VAE', 'CVAE', 'Conv-VAE', 'AE', 'Multimodal', 'PCA', 'Spectral']
mega_rows = []

for ds_key, res in ALL.items():
    for mkey in MODEL_KEYS_ALL:
        r = res['cl'][mkey]['KMeans']
        mega_rows.append({'Dataset': ds_key, 'Model': mkey, 'Type': 'Original',
                          'Sil': r['sil'], 'NMI': r['nmi'], 'ARI': r['ari'], 'Purity': r['purity']})

for ds_key, gr in GMVAE_RESULTS.items():
    m = gr['metrics']
    mega_rows.append({'Dataset': ds_key, 'Model': 'GMVAE', 'Type': 'Extension-1',
                      'Sil': m['sil'], 'NMI': m['nmi'], 'ARI': m['ari'], 'Purity': m['purity']})

for ds_key, cr in CVAE_CON_RESULTS.items():
    m = cr['metrics']
    mega_rows.append({'Dataset': ds_key, 'Model': 'ContrastiveVAE', 'Type': 'Extension-A',
                      'Sil': m['sil'], 'NMI': m['nmi'], 'ARI': m['ari'], 'Purity': m['purity']})

for ds_key, dr in [(k, v) for k, v in DANN_RESULTS.items() if k != '_full']:
    m = dr['metrics']
    mega_rows.append({'Dataset': ds_key, 'Model': 'DANN-VAE', 'Type': 'Extension-B',
                      'Sil': m['sil'], 'NMI': m['nmi'], 'ARI': m['ari'], 'Purity': m['purity']})

df_mega = pd.DataFrame(mega_rows)
df_mega.to_csv(RESULTS / 'mega_comparison.csv', index=False)
plot_mega_heatmap(df_mega, save_path=RESULTS / 'mega_heatmap.png')
print('Saved: mega_comparison.csv  mega_heatmap.png')
print(df_mega.to_string(index=False))

## 7. Advanced Extensions — Final Summary Report

In [ ]:
sep = '=' * 72
print(sep)
print('  ADVANCED EXTENSIONS - FINAL SUMMARY')
print(sep)

print('\n[1] GMVAE vs Standard VAE (KMeans Silhouette):')
for ds_key, gr in GMVAE_RESULTS.items():
    gm_s  = gr['metrics']['sil']
    vae_s = ALL[ds_key]['cl']['MLP-VAE']['KMeans']['sil']
    winner = 'GMVAE' if gm_s > vae_s else 'VAE  '
    print(f'  {ds_key:<8} GMVAE={gm_s:.4f}  VAE={vae_s:.4f}  Winner: {winner}')

print('\n[2] Beta-Sensitivity - Optimal beta per dataset:')
for ds_key, ds_data in BETA_RESULTS.items():
    betas  = sorted(ds_data.keys())
    sils   = [ds_data[b]['metrics']['sil'] for b in betas]
    best_b = betas[int(np.argmax(sils))]
    print(f'  {ds_key:<8} Best beta={best_b:.1f}  (Sil={max(sils):.4f})')

print('\n[3] MIG Disentanglement Scores:')
for ds_key, ds_mig in MIG_RESULTS.items():
    scores = {m: ds_mig[m]['mig'] for m in ds_mig}
    best_m = max(scores, key=scores.__getitem__)
    print(f'  {ds_key:<8} Best: {best_m:<14} MIG={scores[best_m]:.4f}')
    for mname in ('MLP-VAE', 'Beta-VAE', 'GMVAE'):
        if mname in scores:
            print(f'           {mname:<14} MIG={scores[mname]:.4f}')

print('\n[4] Latent Interpolation: SLERP paths saved to interpolation_*.png')

print('\n[5] Cross-Dataset Transfer (Zero-Shot):')
for pair_key, tr in TRANSFER_RESULTS.items():
    qual = ('Good' if tr['retention'] >= 0.7
            else 'Partial' if tr['retention'] >= 0.5
            else 'Poor')
    print(f'  {pair_key:<14} Sil={tr["metrics"]["sil"]:.4f}  '
          f'Retention={tr["retention"]*100:.1f}%  {qual}')

print('\n[A] Contrastive VAE (InfoNCE + beta-VAE):')
for ds_key, cr in CVAE_CON_RESULTS.items():
    vae_s = ALL[ds_key]['cl']['MLP-VAE']['KMeans']['sil']
    con_s = cr['metrics']['sil']
    delta = con_s - vae_s
    qual  = 'Better' if delta > 0.01 else 'Similar' if delta > -0.01 else 'Worse'
    print(f'  {ds_key:<8} lam={cr["best_lam"]:.1f}  Sil={con_s:.4f}  '
          f'vs MLP-VAE={vae_s:.4f}  Delta={delta:+.4f}  {qual}')

print('\n[B] DANN-VAE (Domain Adversarial):')
for ds_key, dr in [(k, v) for k, v in DANN_RESULTS.items() if k != '_full']:
    vae_s  = ALL[ds_key]['cl']['MLP-VAE']['KMeans']['sil']
    dann_s = dr['metrics']['sil']
    delta  = dann_s - vae_s
    qual   = 'Better' if delta > 0.01 else 'Similar' if delta > -0.01 else 'Worse'
    print(f'  {ds_key:<8} Sil={dann_s:.4f}  vs Native={vae_s:.4f}  '
          f'Delta={delta:+.4f}  {qual}')
print(f'  Domain classifier acc: {d_acc:.1f}%  (chance=33.3%)')

print(f'\n  All results saved to: {RESULTS}')
print(sep)